# Quran Word Frequency Counter - Exploration & Discovery

This notebook:
1. Loads and parses the Quranic Arabic Corpus morphology file (v0.4)
2. Explores the data structure and basic statistics
3. Discovers the exact Buckwalter transliteration strings for all 38 target words
4. Validates each discovery by converting to Arabic and checking context

> **Audited 2026-06-11** — see `03_audit_and_full_recount.ipynb` for the full audit, the
> complete method grid required by TASK.txt, and external cross-validation. Corrections made
> here as a result: Izam (bones) now includes the plural عظام tagged under `LEM:EaZiym`
> (count 2 → 15), Hayat gets its variant nouns symmetrically with Mawt, and the
> Akhira/Barr notes carry the corrected numbers.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src.parser import load_morphology, load_prefixes
from src.buckwalter import bw_to_arabic

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 100)

## 1. Load and Parse the Morphology Data

In [2]:
df = load_morphology()
definite_words = load_prefixes()
print(f"Total STEM entries: {len(df):,}")
print(f"Words with definite article (Al+): {len(definite_words):,}")
df.head(10)

Total STEM entries: 77,915
Words with definite article (Al+): 8,377


,chapter,verse,word,segment,form,tag,POS,LEM,ROOT,ASPECT,VOICE,PGN,GENDER,CASE,STATE,FORM_NUM,PCPL,raw_features,NUMBER
0,1,1,1,2,somi,N,N,{som,smw,NaN,NaN,NaN,M,GEN,NaN,NaN,NaN,STEM|POS:N|LEM:{som|ROOT:smw|M|GEN,NaN
1,1,1,2,1,{ll~ahi,PN,PN,{ll~ah,Alh,NaN,NaN,NaN,NaN,GEN,NaN,NaN,NaN,STEM|POS:PN|LEM:{ll~ah|ROOT:Alh|GEN,NaN
2,1,1,3,2,r~aHoma`ni,ADJ,ADJ,r~aHoma`n,rHm,NaN,NaN,MS,NaN,GEN,NaN,NaN,NaN,STEM|POS:ADJ|LEM:r~aHoma`n|ROOT:rHm|MS|GEN,S
3,1,1,4,2,r~aHiymi,ADJ,ADJ,r~aHiym,rHm,NaN,NaN,MS,NaN,GEN,NaN,NaN,NaN,STEM|POS:ADJ|LEM:r~aHiym|ROOT:rHm|MS|GEN,S
4,1,2,1,2,Hamodu,N,N,Hamod,Hmd,NaN,NaN,NaN,M,NOM,NaN,NaN,NaN,STEM|POS:N|LEM:Hamod|ROOT:Hmd|M|NOM,NaN
5,1,2,2,2,l~ahi,PN,PN,{ll~ah,Alh,NaN,NaN,NaN,NaN,GEN,NaN,NaN,NaN,STEM|POS:PN|LEM:{ll~ah|ROOT:Alh|GEN,NaN
6,1,2,3,1,rab~i,N,N,rab~,rbb,NaN,NaN,NaN,M,GEN,NaN,NaN,NaN,STEM|POS:N|LEM:rab~|ROOT:rbb|M|GEN,NaN
7,1,2,4,2,Ea`lamiyna,N,N,Ea`lamiyn,Elm,NaN,NaN,MP,NaN,GEN,NaN,NaN,NaN,STEM|POS:N|LEM:Ea`lamiyn|ROOT:Elm|MP|GEN,P
8,1,3,1,2,r~aHoma`ni,ADJ,ADJ,r~aHoma`n,rHm,NaN,NaN,MS,NaN,GEN,NaN,NaN,NaN,STEM|POS:ADJ|LEM:r~aHoma`n|ROOT:rHm|MS|GEN,S
9,1,3,2,2,r~aHiymi,ADJ,ADJ,r~aHiym,rHm,NaN,NaN,MS,NaN,GEN,NaN,NaN,NaN,STEM|POS:ADJ|LEM:r~aHiym|ROOT:rHm|MS|GEN,S


## 2. Basic Statistics

In [3]:
print("=== POS Distribution ===")
print(df["POS"].value_counts().to_string())
print(f"\n=== Unique roots: {df['ROOT'].nunique():,} ===")
print(f"=== Unique lemmas: {df['LEM'].nunique():,} ===")
print(f"\n=== Number Distribution (from PGN) ===")
print(df["NUMBER"].value_counts().to_string())

=== POS Distribution ===
POS
N       25136
V       19356
P        7679
PN       3911
REL      3575
PRON     3301
NEG      2688
ACC      2283
ADJ      1961
T        1166
DEM      1059
COND     1049
CONJ      756
SUB       684
LOC       669
RES       558
INTG      439
CERT      414
PRO       332
PREV      162
RET       122
EXP       104
INC        90
EXL        66
AMD        65
INT        47
FUT        42
ANS        40
EXH        40
SUR        35
AVR        33
INL        30
SUP        21
IMPN        2

=== Unique roots: 1,642 ===
=== Unique lemmas: 4,832 ===

=== Number Distribution (from PGN) ===
NUMBER
P    18877
S    16629
D      426


## 3. Discovery: Find Exact Buckwalter Strings for All 38 Words

For each target word, we search by root (and/or lemma pattern) and display all lemmas found, with Arabic conversion for human verification.

### Helper Functions

In [4]:
def discover_by_root(root: str, pos_filter: list[str] | None = None) -> pd.DataFrame:
    """Find all lemmas under a given root, with counts and Arabic display."""
    mask = df["ROOT"] == root
    if pos_filter:
        mask &= df["POS"].isin(pos_filter)
    subset = df[mask]
    if subset.empty:
        print(f"  No entries found for ROOT:{root}")
        return pd.DataFrame()
    
    counts = subset.groupby(["LEM", "POS"]).size().reset_index(name="count")
    counts["arabic"] = counts["LEM"].apply(bw_to_arabic)
    counts = counts.sort_values("count", ascending=False).reset_index(drop=True)
    return counts


def discover_by_lemma(lemma: str) -> pd.DataFrame:
    """Find all entries with an exact lemma match, with POS and number breakdown."""
    subset = df[df["LEM"] == lemma]
    if subset.empty:
        print(f"  No entries found for LEM:{lemma}")
        return pd.DataFrame()
    
    print(f"  Total count: {len(subset)}")
    print(f"  Arabic: {bw_to_arabic(lemma)}")
    
    pos_counts = subset["POS"].value_counts()
    print(f"  By POS: {pos_counts.to_dict()}")
    
    num_counts = subset["NUMBER"].value_counts()
    print(f"  By Number: {num_counts.to_dict()}")
    
    return subset


def discover_no_root(lemma_pattern: str) -> pd.DataFrame:
    """For proper nouns with no root - search by lemma pattern."""
    mask = df["LEM"].str.contains(lemma_pattern, na=False)
    subset = df[mask]
    if subset.empty:
        print(f"  No entries found matching LEM pattern: {lemma_pattern}")
        return pd.DataFrame()
    
    counts = subset.groupby(["LEM", "POS"]).size().reset_index(name="count")
    counts["arabic"] = counts["LEM"].apply(bw_to_arabic)
    return counts


def has_definite_article(row) -> bool:
    """Check if a word has the definite article prefix Al+."""
    return (row["chapter"], row["verse"], row["word"]) in definite_words


def show_sample_verses(lemma: str, n: int = 5):
    """Show sample verse locations for a lemma, with definite article info."""
    subset = df[df["LEM"] == lemma].head(n)
    for _, row in subset.iterrows():
        has_al = "AL+" if has_definite_article(row) else "    "
        print(f"  ({row['chapter']:>3}:{row['verse']:>3}:{row['word']}) {has_al} form={row['form']} [{bw_to_arabic(row['form'])}] POS={row['POS']}")


def count_with_article(lemma: str, pos_filter: list[str] | None = None) -> dict:
    """Count a lemma, broken down by definite/indefinite and POS."""
    mask = df["LEM"] == lemma
    if pos_filter:
        mask &= df["POS"].isin(pos_filter)
    subset = df[mask].copy()
    subset["has_al"] = subset.apply(has_definite_article, axis=1)
    
    total = len(subset)
    with_al = subset["has_al"].sum()
    without_al = total - with_al
    
    return {
        "total": total,
        "with_al": int(with_al),
        "without_al": int(without_al),
        "by_pos": subset["POS"].value_counts().to_dict(),
        "by_number": subset["NUMBER"].value_counts().to_dict(),
    }

### 3.1 Dunya & Akhira (This World & Hereafter)

In [5]:
print("=== DUNYA (this world) - ROOT:dnw ===")
display(discover_by_root("dnw"))
print("\nTarget lemma: d~unoyaA")
print(count_with_article("d~unoyaA"))
show_sample_verses("d~unoyaA")

print("\n\n=== AKHIRA (hereafter) - ROOT:Axr ===")
display(discover_by_root("Axr"))
print("\nTarget lemma: A^xir")
print(count_with_article("A^xir"))
show_sample_verses("A^xir")

=== DUNYA (this world) - ROOT:dnw ===


,LEM,POS,count,arabic
0,d~unoyaA,ADJ,74,دُّنْيَا
1,d~unoyaA,N,41,دُّنْيَا
2,>adonaY`,N,10,أَدْنَىٰ
3,daAniyap,N,2,دَانِيَة
4,danaA,V,2,دَنَا
5,>adonaY`,ADJ,1,أَدْنَىٰ
6,>adonaY`,T,1,أَدْنَىٰ
7,daAn,N,1,دَان
8,daAniyap,ADJ,1,دَانِيَة



Target lemma: d~unoyaA
{'total': 115, 'with_al': 115, 'without_al': 0, 'by_pos': {'ADJ': 74, 'N': 41}, 'by_number': {'S': 115}}
  (  2: 85:38) AL+ form=d~unoyaA [دُّنْيَا] POS=ADJ
  (  2: 86:5) AL+ form=d~unoyaA [دُّنْيَا] POS=ADJ
  (  2:114:24) AL+ form=d~unoyaA [دُّنْيَا] POS=N
  (  2:130:13) AL+ form=d~unoyaA [دُّنْيَا] POS=N
  (  2:200:18) AL+ form=d~unoyaA [دُّنْيَا] POS=N


=== AKHIRA (hereafter) - ROOT:Axr ===


,LEM,POS,count,arabic
0,A^xir,N,133,ا^خِر
1,A^xar,N,44,ا^خَر
2,A^xar,ADJ,26,ا^خَر
3,A^xir,ADJ,21,ا^خِر
4,>ax~ara,V,15,أَخَّرَ
5,yasota>oxiru,V,6,يَسْتَأْخِرُ
6,ta>ax~ara,V,3,تَأَخَّرَ
7,A^xir,T,1,ا^خِر
8,musota_#oxiriyn,N,1,مُسْتَـ#ْخِرِين



Target lemma: A^xir
{'total': 155, 'with_al': 152, 'without_al': 3, 'by_pos': {'N': 133, 'ADJ': 21, 'T': 1}, 'by_number': {'S': 145, 'P': 10}}
  (  2:  4:10) AL+ form='aAxirapi [ءَاخِرَةِ] POS=N
  (  2:  8:8) AL+ form='aAxiri [ءَاخِرِ] POS=ADJ
  (  2: 62:12) AL+ form='aAxiri [ءَاخِرِ] POS=ADJ
  (  2: 86:6) AL+ form='aAxirapi [ءَاخِرَةِ] POS=N
  (  2: 94:6) AL+ form='aAxirapu [ءَاخِرَةُ] POS=ADJ


### 3.2 Malak (Angel) & Shaytan (Satan)

In [6]:
print("=== MALAK (angel) - ROOT:mlk ===")
print("ROOT:mlk includes king/kingdom words. Filter by LEM:malak for angels only.\n")
display(discover_by_root("mlk"))
print("\nTarget lemma: malak (angel)")
print(count_with_article("malak"))
show_sample_verses("malak")

print("\n\n=== SHAYTAN (satan) - ROOT:$Tn ===")
display(discover_by_root("$Tn"))
print("\nTarget lemma: $ayoTa`n")
print(count_with_article("$ayoTa`n"))
show_sample_verses("$ayoTa`n")

=== MALAK (angel) - ROOT:mlk ===
ROOT:mlk includes king/kingdom words. Filter by LEM:malak for angels only.



,LEM,POS,count,arabic
0,malak,N,88,مَلَك
1,mulok,N,48,مُلْك
2,malakato,V,44,مَلَكَتْ
3,malik,N,13,مَلِك
4,malakuwt,N,4,مَلَكُوت
5,ma`lik,N,3,مَٰلِك
6,malik,ADJ,2,مَلِك
7,ma`lik2,PN,1,مَٰلِك2
8,maliyk,N,1,مَلِيك
9,malok,N,1,مَلْك



Target lemma: malak (angel)
{'total': 88, 'with_al': 65, 'without_al': 23, 'by_pos': {'N': 88}, 'by_number': {'P': 73, 'D': 2}}
  (  2: 30:4) AL+ form=mala`^}ikapi [مَلَٰ^ئِكَةِ] POS=N
  (  2: 31:8) AL+ form=mala`^}ikapi [مَلَٰ^ئِكَةِ] POS=N
  (  2: 34:3) AL+ form=mala`^}ikapi [مَلَٰ^ئِكَةِ] POS=N
  (  2: 98:5)      form=mala`^}ikati [مَلَٰ^ئِكَتِ] POS=N
  (  2:102:20) AL+ form=malakayoni [مَلَكَيْنِ] POS=N


=== SHAYTAN (satan) - ROOT:$Tn ===


,LEM,POS,count,arabic
0,$ayoTa`n,PN,80,شَيْطَٰن
1,$ayoTa`n,N,8,شَيْطَٰن



Target lemma: $ayoTa`n
{'total': 88, 'with_al': 80, 'without_al': 8, 'by_pos': {'PN': 80, 'N': 8}, 'by_number': {'P': 18}}
  (  2: 14:10)      form=$aya`Tiyni [شَيَٰطِينِ] POS=N
  (  2: 36:2) AL+ form=$~ayoTa`nu [شَّيْطَٰنُ] POS=PN
  (  2:102:4) AL+ form=$~aya`Tiynu [شَّيَٰطِينُ] POS=PN
  (  2:102:12) AL+ form=$~aya`Tiyna [شَّيَٰطِينَ] POS=PN
  (  2:168:12) AL+ form=$~ayoTa`ni [شَّيْطَٰنِ] POS=PN


In [7]:
# --- 3.3 Hayat (life) & Mawt (death) ---
print("=== HAYAT (life) - ROOT:Hyy ===")
display(discover_by_root("Hyy"))
print("\nTarget lemma: Hayaw`p (حياة = life)")
print(count_with_article("Hayaw`p"))
show_sample_verses("Hayaw`p")

print("\n\n=== MAWT (death) - ROOT:mwt ===")
display(discover_by_root("mwt"))
print("\nTarget lemma: mawot (موت = death)")
print(count_with_article("mawot"))
show_sample_verses("mawot")

=== HAYAT (life) - ROOT:Hyy ===


,LEM,POS,count,arabic
0,Hayaw`p,N,76,حَيَوٰة
1,>aHoyaA,V,51,أَحْيَا
2,Hay~,N,23,حَيّ
3,yasotaHoYi.^,V,9,يَسْتَحْىِ.^
4,HaY~a,V,7,حَىَّ
5,taHiy~ap,N,6,تَحِيَّة
6,Hay~a,V,4,حَيَّ
7,muHoY,N,2,مُحْى
8,m~aHoyaA,N,2,مَّحْيَا
9,HayawaAn,N,1,حَيَوَان



Target lemma: Hayaw`p (حياة = life)
{'total': 76, 'with_al': 67, 'without_al': 9, 'by_pos': {'N': 76}, 'by_number': {}}
  (  2: 85:37) AL+ form=Hayaw`pi [حَيَوٰةِ] POS=N
  (  2: 86:4) AL+ form=Hayaw`pa [حَيَوٰةَ] POS=N
  (  2: 96:5)      form=Hayaw`pK [حَيَوٰةٍ] POS=N
  (  2:179:4)      form=Hayaw`pN [حَيَوٰةٌ] POS=N
  (  2:204:7) AL+ form=Hayaw`pi [حَيَوٰةِ] POS=N


=== MAWT (death) - ROOT:mwt ===


,LEM,POS,count,arabic
0,mawot,N,50,مَوْت
1,m~aAta,V,39,مَّاتَ
2,m~ay~it,N,34,مَّيِّت
3,>amaAta,V,21,أَمَاتَ
4,mayotap,N,5,مَيْتَة
5,m~ayot,ADJ,4,مَّيْت
6,m~ay~it,ADJ,4,مَّيِّت
7,mamaAt,N,3,مَمَات
8,mawotat,N,3,مَوْتَت
9,mayotap,ADJ,1,مَيْتَة



Target lemma: mawot (موت = death)
{'total': 50, 'with_al': 35, 'without_al': 15, 'by_pos': {'N': 50}, 'by_number': {}}
  (  2: 19:16) AL+ form=mawoti [مَوْتِ] POS=N
  (  2: 56:5)      form=mawoti [مَوْتِ] POS=N
  (  2: 94:14) AL+ form=mawota [مَوْتَ] POS=N
  (  2:133:7) AL+ form=mawotu [مَوْتُ] POS=N
  (  2:164:28)      form=mawoti [مَوْتِ] POS=N


In [8]:
# --- 3.4 Rajul (man) & Imra'a (woman) ---
print("=== RAJUL (man) - ROOT:rjl ===")
display(discover_by_root("rjl"))
print("\nTarget lemma: rajul (رجل = man)")
print(count_with_article("rajul"))
show_sample_verses("rajul")
# Also check related lemmas for completeness
print("\nOther lemmas under ROOT:rjl:")
for lem in ["rijaAl", "rijol", "rajil"]:
    c = len(df[df["LEM"] == lem])
    print(f"  {lem} [{bw_to_arabic(lem)}]: {c}")

print("\n\n=== IMRA'A (woman) - ROOT:mrA ===")
display(discover_by_root("mrA"))
print("\nTarget lemma: {mora>at (امرأة = woman)")
print(count_with_article("{mora>at"))
show_sample_verses("{mora>at")

=== RAJUL (man) - ROOT:rjl ===


,LEM,POS,count,arabic
0,rajul,N,29,رَجُل
1,rijaAl,N,28,رِجَال
2,rijol,N,15,رِجْل
3,rajil,N,1,رَجِل



Target lemma: rajul (رجل = man)
{'total': 29, 'with_al': 0, 'without_al': 29, 'by_pos': {'N': 29}, 'by_number': {'D': 5}}
  (  2:282:59)      form=rajulayoni [رَجُلَيْنِ] POS=N
  (  2:282:60)      form=rajulN [رَجُلٌ] POS=N
  (  4: 12:52)      form=rajulN [رَجُلٌ] POS=N
  (  5: 23:2)      form=rajulaAni [رَجُلَانِ] POS=N
  (  6:  9:5)      form=rajulFA [رَجُلًا] POS=N

Other lemmas under ROOT:rjl:
  rijaAl [رِجَال]: 28
  rijol [رِجْل]: 15
  rajil [رَجِل]: 1


=== IMRA'A (woman) - ROOT:mrA ===


,LEM,POS,count,arabic
0,{mora>at,N,26,ٱمْرَأَت
1,{mori},N,5,ٱمْرِئ
2,maro',N,4,مَرْء
3,m~ariy^_#,ADJ,1,مَّرِي^ـ#
4,{mora>,N,1,ٱمْرَأ
5,{moru&NA,N,1,ٱمْرُؤٌا



Target lemma: {mora>at (امرأة = woman)
{'total': 26, 'with_al': 0, 'without_al': 26, 'by_pos': {'N': 26}, 'by_number': {'D': 2}}
  (  2:282:61)      form={mora>ataAni [ٱمْرَأَتَانِ] POS=N
  (  3: 35:3)      form={mora>atu [ٱمْرَأَتُ] POS=N
  (  3: 40:10)      form={mora>ati [ٱمْرَأَتِ] POS=N
  (  4: 12:56)      form={mora>apN [ٱمْرَأَةٌ] POS=N
  (  4:128:2)      form={mora>apN [ٱمْرَأَةٌ] POS=N


In [9]:
# --- 3.5 Shahr (month) & Yawm (day) ---
print("=== SHAHR (month) - ROOT:$hr ===")
display(discover_by_root("$hr"))
print("\nTarget lemma: $ahor (شهر = month)")
print(count_with_article("$ahor"))
show_sample_verses("$ahor")

print("\n\n=== YAWM (day) - ROOT:ywm ===")
display(discover_by_root("ywm"))
print("\nTarget lemma: yawom (يوم = day)")
print(count_with_article("yawom"))
show_sample_verses("yawom")

=== SHAHR (month) - ROOT:$hr ===


,LEM,POS,count,arabic
0,$ahor,N,21,شَهْر



Target lemma: $ahor (شهر = month)
{'total': 21, 'with_al': 8, 'without_al': 13, 'by_pos': {'N': 21}, 'by_number': {'P': 7, 'D': 2}}
  (  2:185:1)      form=$ahoru [شَهْرُ] POS=N
  (  2:185:16) AL+ form=$~ahora [شَّهْرَ] POS=N
  (  2:194:1) AL+ form=$~ahoru [شَّهْرُ] POS=N
  (  2:194:3) AL+ form=$~ahori [شَّهْرِ] POS=N
  (  2:197:2)      form=>a$ohurN [أَشْهُرٌ] POS=N


=== YAWM (day) - ROOT:ywm ===


,LEM,POS,count,arabic
0,yawom,N,325,يَوْم
1,yawom,T,80,يَوْم



Target lemma: yawom (يوم = day)
{'total': 405, 'with_al': 77, 'without_al': 328, 'by_pos': {'N': 325, 'T': 80}, 'by_number': {'P': 27, 'D': 3}}


  (  1:  4:2)      form=yawomi [يَوْمِ] POS=N
  (  2:  8:7) AL+ form=yawomi [يَوْمِ] POS=N
  (  2: 48:2)      form=yawomFA [يَوْمًا] POS=N
  (  2: 62:11) AL+ form=yawomi [يَوْمِ] POS=N
  (  2: 80:6)      form=>ay~aAmFA [أَيَّامًا] POS=N


In [10]:
# --- 3.6 Bahr (sea) & Barr (land) ---
print("=== BAHR (sea) - ROOT:bHr ===")
display(discover_by_root("bHr"))
print("\nTarget lemma: baHor (بحر = sea)")
print(count_with_article("baHor"))
show_sample_verses("baHor")

print("\n\n=== BARR (land) - ROOT:brr ===")
display(discover_by_root("brr"))
print("\nTarget lemma: bar~ (بر = land)")
print("Note: bar~ also means 'righteousness/piety'. Review samples for semantic split.")
print(count_with_article("bar~"))
show_sample_verses("bar~", n=10)

=== BAHR (sea) - ROOT:bHr ===


,LEM,POS,count,arabic
0,baHor,N,41,بَحْر
1,baHiyrap,N,1,بَحِيرَة



Target lemma: baHor (بحر = sea)
{'total': 41, 'with_al': 39, 'without_al': 2, 'by_pos': {'N': 41}, 'by_number': {'D': 5, 'P': 3}}
  (  2: 50:4) AL+ form=baHora [بَحْرَ] POS=N
  (  2:164:13) AL+ form=baHori [بَحْرِ] POS=N
  (  5: 96:4) AL+ form=baHori [بَحْرِ] POS=N
  (  6: 59:12) AL+ form=baHori [بَحْرِ] POS=N
  (  6: 63:7) AL+ form=baHori [بَحْرِ] POS=N


=== BARR (land) - ROOT:brr ===


,LEM,POS,count,arabic
0,bar~,N,21,بَرّ
1,bir~,N,8,بِرّ
2,tabar~u,V,2,تَبَرُّ
3,bar~,ADJ,1,بَرّ



Target lemma: bar~ (بر = land)
Note: bar~ also means 'righteousness/piety'. Review samples for semantic split.
{'total': 22, 'with_al': 19, 'without_al': 3, 'by_pos': {'N': 21, 'ADJ': 1}, 'by_number': {'P': 7}}
  (  3:193:20) AL+ form=>aboraAri [أَبْرَارِ] POS=N
  (  3:198:21) AL+ form=>aboraAri [أَبْرَارِ] POS=N
  (  5: 96:12) AL+ form=bar~i [بَرِّ] POS=N
  (  6: 59:11) AL+ form=bar~i [بَرِّ] POS=N
  (  6: 63:6) AL+ form=bar~i [بَرِّ] POS=N
  (  6: 97:10) AL+ form=bar~i [بَرِّ] POS=N
  ( 10: 22:5) AL+ form=bar~i [بَرِّ] POS=N
  ( 17: 67:14) AL+ form=bar~i [بَرِّ] POS=N
  ( 17: 68:6) AL+ form=bar~i [بَرِّ] POS=N
  ( 17: 70:7) AL+ form=bar~i [بَرِّ] POS=N


In [11]:
# --- 3.7 Jannah (heaven/garden) & Jahannam (hell) ---
print("=== JANNAH (heaven/garden) - ROOT:jnn ===")
display(discover_by_root("jnn"))
print("\nTarget lemma: jan~ap (جنة = garden/heaven)")
print(count_with_article("jan~ap"))
show_sample_verses("jan~ap")

print("\n\n=== JAHANNAM (hell) - no ROOT, proper noun ===")
display(discover_no_root("jahan~am"))
print("\nTarget lemma: jahan~am (جهنم)")
print(count_with_article("jahan~am"))
show_sample_verses("jahan~am")

=== JANNAH (heaven/garden) - ROOT:jnn ===


,LEM,POS,count,arabic
0,jan~ap,N,96,جَنَّة
1,jan~ap,PN,51,جَنَّة
2,jin~,N,22,جِنّ
3,majonuwn,N,11,مَجْنُون
4,jin~ap,N,10,جِنَّة
5,jaA^n~,N,7,جَا^نّ
6,jun~ap,N,2,جُنَّة
7,>ajin~ap,N,1,أَجِنَّة
8,jan~a,V,1,جَنَّ



Target lemma: jan~ap (جنة = garden/heaven)
{'total': 147, 'with_al': 55, 'without_al': 92, 'by_pos': {'N': 96, 'PN': 51}, 'by_number': {'P': 71, 'D': 8}}
  (  2: 25:8)      form=jan~a`tK [جَنَّٰتٍ] POS=N
  (  2: 35:6) AL+ form=jan~apa [جَنَّةَ] POS=PN
  (  2: 82:7) AL+ form=jan~api [جَنَّةِ] POS=PN
  (  2:111:4) AL+ form=jan~apa [جَنَّةَ] POS=PN
  (  2:214:5) AL+ form=jan~apa [جَنَّةَ] POS=PN


=== JAHANNAM (hell) - no ROOT, proper noun ===


,LEM,POS,count,arabic
0,jahan~am,PN,77,جَهَنَّم



Target lemma: jahan~am (جهنم)
{'total': 77, 'with_al': 0, 'without_al': 77, 'by_pos': {'PN': 77}, 'by_number': {}}


  (  2:206:10)      form=jahan~amu [جَهَنَّمُ] POS=PN
  (  3: 12:7)      form=jahan~ama [جَهَنَّمَ] POS=PN
  (  3:162:11)      form=jahan~amu [جَهَنَّمُ] POS=PN
  (  3:197:5)      form=jahan~amu [جَهَنَّمُ] POS=PN
  (  4: 55:10)      form=jahan~ama [جَهَنَّمَ] POS=PN


In [12]:
# --- 3.8 Harr (hot) & Bard (cold) ---
print("=== HARR (hot/heat) - ROOT:Hrr ===")
display(discover_by_root("Hrr"))
print("\nTarget lemma: Har~ (حر = heat/hot)")
print(count_with_article("Har~"))
show_sample_verses("Har~")
print("\nAlso checking Haruwr (حرور = scorching heat):")
print(f"  count: {len(df[df['LEM'] == 'Haruwr'])}")

print("\n\n=== BARD (cold) - ROOT:brd ===")
display(discover_by_root("brd"))
print("\nTarget lemma: barod (برد = cold)")
print(count_with_article("barod"))
show_sample_verses("barod")

=== HARR (hot/heat) - ROOT:Hrr ===


,LEM,POS,count,arabic
0,taHoriyr,N,5,تَحْرِير
1,Hariyr,N,3,حَرِير
2,Har~,N,3,حَرّ
3,Hur~,N,2,حُرّ
4,Haruwr,N,1,حَرُور
5,muHar~ar,N,1,مُحَرَّر



Target lemma: Har~ (حر = heat/hot)
{'total': 3, 'with_al': 2, 'without_al': 1, 'by_pos': {'N': 3}, 'by_number': {}}
  (  9: 81:19) AL+ form=Har~i [حَرِّ] POS=N
  (  9: 81:24)      form=Har~FA [حَرًّا] POS=N
  ( 16: 81:16) AL+ form=Har~a [حَرَّ] POS=N

Also checking Haruwr (حرور = scorching heat):
  count: 1


=== BARD (cold) - ROOT:brd ===


,LEM,POS,count,arabic
0,baArid,N,2,بَارِد
1,barod,N,2,بَرْد
2,barad,N,1,بَرَد



Target lemma: barod (برد = cold)
{'total': 2, 'with_al': 0, 'without_al': 2, 'by_pos': {'N': 2}, 'by_number': {}}
  ( 21: 69:4)      form=barodFA [بَرْدًا] POS=N
  ( 78: 24:4)      form=barodFA [بَرْدًا] POS=N


In [13]:
# --- 3.9 Zakat & Baraka (blessing) ---
print("=== ZAKAT - ROOT:zkw ===")
display(discover_by_root("zkw"))
print("\nTarget lemma: zakaw`p (زكاة = zakat/purification)")
print(count_with_article("zakaw`p"))
show_sample_verses("zakaw`p")

print("\n\n=== BARAKA (blessing) - ROOT:brk ===")
display(discover_by_root("brk"))
print("\nTarget lemma: ba`raka (بارك = blessed)")
print(count_with_article("ba`raka"))
show_sample_verses("ba`raka")
print("\nAlso baraka`t (بركات = blessings):")
print(count_with_article("baraka`t"))
show_sample_verses("baraka`t")

=== ZAKAT - ROOT:zkw ===


,LEM,POS,count,arabic
0,zakaw`p,N,32,زَكَوٰة
1,zak~aY`,V,12,زَكَّىٰ
2,tazak~aY`,V,8,تَزَكَّىٰ
3,>azokaY`,N,4,أَزْكَىٰ
4,zakaY`,V,1,زَكَىٰ
5,zakiy~,ADJ,1,زَكِيّ
6,zakiy~ap,N,1,زَكِيَّة



Target lemma: zakaw`p (زكاة = zakat/purification)
{'total': 32, 'with_al': 29, 'without_al': 3, 'by_pos': {'N': 32}, 'by_number': {}}
  (  2: 43:4) AL+ form=z~akaw`pa [زَّكَوٰةَ] POS=N
  (  2: 83:22) AL+ form=z~akaw`pa [زَّكَوٰةَ] POS=N
  (  2:110:4) AL+ form=z~akaw`pa [زَّكَوٰةَ] POS=N
  (  2:177:35) AL+ form=z~akaw`pa [زَّكَوٰةَ] POS=N
  (  2:277:9) AL+ form=z~akaw`pa [زَّكَوٰةَ] POS=N


=== BARAKA (blessing) - ROOT:brk ===


,LEM,POS,count,arabic
0,tabaAraka,V,9,تَبَارَكَ
1,ba`raka,V,8,بَٰرَكَ
2,mubaArak,N,6,مُبَارَك
3,baraka`t,N,3,بَرَكَٰت
4,m~uba`rakap,N,3,مُّبَٰرَكَة
5,mubaArak,ADJ,2,مُبَارَك
6,m~uba`rakap,ADJ,1,مُّبَٰرَكَة



Target lemma: ba`raka (بارك = blessed)
{'total': 8, 'with_al': 0, 'without_al': 8, 'by_pos': {'V': 8}, 'by_number': {'P': 6, 'S': 2}}


  (  7:137:10)      form=ba`rako [بَٰرَكْ] POS=V
  ( 17:  1:13)      form=ba`rako [بَٰرَكْ] POS=V
  ( 21: 71:6)      form=ba`rako [بَٰرَكْ] POS=V
  ( 21: 81:9)      form=ba`rako [بَٰرَكْ] POS=V
  ( 27:  8:5)      form=buwrika [بُورِكَ] POS=V

Also baraka`t (بركات = blessings):
{'total': 3, 'with_al': 0, 'without_al': 3, 'by_pos': {'N': 3}, 'by_number': {'P': 3}}
  (  7: 96:9)      form=baraka`tK [بَرَكَٰتٍ] POS=N
  ( 11: 48:6)      form=baraka`tK [بَرَكَٰتٍ] POS=N
  ( 11: 73:8)      form=baraka`tu [بَرَكَٰتُ] POS=N


In [14]:
# --- 3.10 Qaala (said) - ROOT:qwl - WITH VERB FORM BREAKDOWN ---
print("=== QAALA (said) - ROOT:qwl ===")
display(discover_by_root("qwl"))

print("\n--- Verb breakdown for LEM:qaAla ---")
qaala = df[(df["LEM"] == "qaAla") & (df["POS"] == "V")]
print(f"Total verb occurrences: {len(qaala)}")

# Breakdown by ASPECT (perfect/imperfect/imperative)
print("\nBy Aspect:")
print(qaala["ASPECT"].value_counts().to_string())

# Breakdown by VOICE (active/passive)
print("\nBy Voice:")
print(qaala["VOICE"].value_counts().to_string())

# Breakdown by ASPECT + VOICE + PGN
print("\nDetailed breakdown (Aspect x Voice x Person-Gender-Number):")
breakdown = qaala.groupby(["ASPECT", "VOICE", "PGN"]).size().reset_index(name="count")
breakdown = breakdown.sort_values("count", ascending=False).reset_index(drop=True)
display(breakdown)

# Also show the noun/verbal noun forms
print("\n--- Non-verb forms under ROOT:qwl ---")
print(f"  qawol (قول = speech/saying): {len(df[df['LEM'] == 'qawol'])}")
print(f"  qaA^}}il (قائل = one who says): {len(df[df['LEM'] == 'qaA^}il'])}")
print(f"  qiyl (قيل = it was said): {len(df[df['LEM'] == 'qiyl'])}")
print(f"  >aqaAwiyl (أقاويل = false sayings): {len(df[df['LEM'] == '>aqaAwiyl'])}")

=== QAALA (said) - ROOT:qwl ===


,LEM,POS,count,arabic
0,qaAla,V,1618,قَالَ
1,qawol,N,92,قَوْل
2,qaA^}il,N,5,قَا^ئِل
3,qiyl,N,4,قِيل
4,taqaw~ala,V,2,تَقَوَّلَ
5,>aqaAwiyl,N,1,أَقَاوِيل



--- Verb breakdown for LEM:qaAla ---
Total verb occurrences: 1618

By Aspect:
ASPECT
PERF    1004
IMPV     349
IMPF     265

By Voice:
VOICE
PASS    52

Detailed breakdown (Aspect x Voice x Person-Gender-Number):


,ASPECT,VOICE,PGN,count
0,PERF,PASS,3MS,49
1,IMPF,PASS,3MS,3



--- Non-verb forms under ROOT:qwl ---


  qawol (قول = speech/saying): 92
  qaA^}il (قائل = one who says): 5
  qiyl (قيل = it was said): 4


  >aqaAwiyl (أقاويل = false sayings): 1


In [15]:
# --- 3.11 Adhab (punishment) & Rahma (mercy) & Maghfira (forgiveness) ---
print("=== ADHAB (punishment) - ROOT:E*b ===")
display(discover_by_root("E*b"))
print("\nTarget lemma: Ea*aAb (عذاب = punishment/torment)")
print(count_with_article("Ea*aAb"))
show_sample_verses("Ea*aAb")

print("\n\n=== RAHMA (mercy) - ROOT:rHm ===")
display(discover_by_root("rHm"))
print("\nTarget lemma: raHomap (رحمة = mercy)")
print(count_with_article("raHomap"))
show_sample_verses("raHomap")

print("\n\n=== MAGHFIRA (forgiveness) - ROOT:gfr ===")
display(discover_by_root("gfr"))
print("\nTarget lemma: m~agofirap (مغفرة = forgiveness)")
print(count_with_article("m~agofirap"))
show_sample_verses("m~agofirap")

=== ADHAB (punishment) - ROOT:E*b ===


,LEM,POS,count,arabic
0,Ea*aAb,N,322,عَذَاب
1,Ea*~aba,V,41,عَذَّبَ
2,muEa*~abiyn,N,4,مُعَذَّبِين
3,muEa*~ib,N,4,مُعَذِّب
4,Ea*ob,N,2,عَذْب



Target lemma: Ea*aAb (عذاب = punishment/torment)
{'total': 322, 'with_al': 92, 'without_al': 230, 'by_pos': {'N': 322}, 'by_number': {}}
  (  2:  7:11)      form=Ea*aAbN [عَذَابٌ] POS=N
  (  2: 10:8)      form=Ea*aAbN [عَذَابٌ] POS=N
  (  2: 49:8) AL+ form=Ea*aAbi [عَذَابِ] POS=N
  (  2: 85:44) AL+ form=Ea*aAbi [عَذَابِ] POS=N
  (  2: 86:10) AL+ form=Ea*aAbu [عَذَابُ] POS=N


=== RAHMA (mercy) - ROOT:rHm ===


,LEM,POS,count,arabic
0,raHomap,N,114,رَحْمَة
1,r~aHiym,ADJ,112,رَّحِيم
2,r~aHoma`n,N,45,رَّحْمَٰن
3,r~aHima,V,28,رَّحِمَ
4,>aroHaAm,N,12,أَرْحَام
5,r~aHoma`n,ADJ,12,رَّحْمَٰن
6,r~a`Himiyn,N,6,رَّٰحِمِين
7,>aroHam,N,4,أَرْحَم
8,r~aHiym,N,4,رَّحِيم
9,maroHamap,N,1,مَرْحَمَة



Target lemma: raHomap (رحمة = mercy)


{'total': 114, 'with_al': 6, 'without_al': 108, 'by_pos': {'N': 114}, 'by_number': {}}
  (  2: 64:10)      form=raHomatu [رَحْمَتُ] POS=N
  (  2:105:19)      form=raHomati [رَحْمَتِ] POS=N
  (  2:157:6)      form=raHomapN [رَحْمَةٌ] POS=N
  (  2:178:30)      form=raHomapN [رَحْمَةٌ] POS=N
  (  2:218:12)      form=raHomata [رَحْمَتَ] POS=N


=== MAGHFIRA (forgiveness) - ROOT:gfr ===


,LEM,POS,count,arabic
0,gafara,V,65,غَفَرَ
1,gafuwr,N,62,غَفُور
2,{sotagofara,V,40,ٱسْتَغْفَرَ
3,gafuwr,ADJ,29,غَفُور
4,m~agofirap,N,28,مَّغْفِرَة
5,gaf~aAr,ADJ,3,غَفَّار
6,gaAfir,N,2,غَافِر
7,gaf~aAr,N,2,غَفَّار
8,guforaAn,N,1,غُفْرَان
9,musotagofiriyn,N,1,مُسْتَغْفِرِين



Target lemma: m~agofirap (مغفرة = forgiveness)
{'total': 28, 'with_al': 4, 'without_al': 24, 'by_pos': {'N': 28}, 'by_number': {}}
  (  2:175:7) AL+ form=magofirapi [مَغْفِرَةِ] POS=N
  (  2:221:33) AL+ form=magofirapi [مَغْفِرَةِ] POS=N
  (  2:263:3)      form=magofirapN [مَغْفِرَةٌ] POS=N
  (  2:268:8)      form=m~agofirapF [مَّغْفِرَةً] POS=N
  (  3:133:3)      form=magofirapK [مَغْفِرَةٍ] POS=N


In [16]:
# --- 3.12 Ghani (rich) & Faqir (poor) ---
print("=== GHANI (rich) - ROOT:gny ===")
display(discover_by_root("gny"))
print("\nTarget lemma: ganiY~ (غني = rich/self-sufficient)")
print(count_with_article("ganiY~"))
show_sample_verses("ganiY~")

print("\n\n=== FAQIR (poor) - ROOT:fqr ===")
display(discover_by_root("fqr"))
print("\nTarget lemma: faqiyr (فقير = poor)")
print(count_with_article("faqiyr"))
show_sample_verses("faqiyr")

=== GHANI (rich) - ROOT:gny ===


,LEM,POS,count,arabic
0,>agonato,V,28,أَغْنَتْ
1,ganiY~,N,20,غَنِىّ
2,>agonaY`,V,15,أَغْنَىٰ
3,ganiY~,ADJ,4,غَنِىّ
4,{sotagonaY`,V,4,ٱسْتَغْنَىٰ
5,m~ugonuwn,N,2,مُّغْنُون



Target lemma: ganiY~ (غني = rich/self-sufficient)


{'total': 24, 'with_al': 9, 'without_al': 15, 'by_pos': {'N': 20, 'ADJ': 4}, 'by_number': {'S': 20, 'P': 4}}
  (  2:263:10)      form=ganiY~N [غَنِىٌّ] POS=N
  (  2:267:28)      form=ganiY~N [غَنِىٌّ] POS=N
  (  2:273:14)      form=>agoniyaA^'a [أَغْنِيَا^ءَ] POS=N
  (  3: 97:23)      form=ganiY~N [غَنِىٌّ] POS=N
  (  3:181:11)      form=>agoniyaA^'u [أَغْنِيَا^ءُ] POS=N


=== FAQIR (poor) - ROOT:fqr ===


,LEM,POS,count,arabic
0,faqiyr,N,10,فَقِير
1,faqiyr,ADJ,2,فَقِير
2,faAqirap,N,1,فَاقِرَة
3,faqor,N,1,فَقْر



Target lemma: faqiyr (فقير = poor)
{'total': 12, 'with_al': 7, 'without_al': 5, 'by_pos': {'N': 10, 'ADJ': 2}, 'by_number': {'P': 7, 'S': 5}}
  (  2:271:9) AL+ form=fuqaraA^'a [فُقَرَا^ءَ] POS=N
  (  2:273:1) AL+ form=fuqaraA^'i [فُقَرَا^ءِ] POS=N
  (  3:181:9)      form=faqiyrN [فَقِيرٌ] POS=N
  (  4:  6:26)      form=faqiyrFA [فَقِيرًا] POS=N
  (  4:135:19)      form=faqiyrFA [فَقِيرًا] POS=N


In [17]:
# --- 3.13 Hasana (good deed) & Sayyi'a (bad deed) ---
print("=== HASANA (good deed) - ROOT:Hsn ===")
display(discover_by_root("Hsn"))
print("\nTarget lemma: Hasanap (حسنة = good deed)")
print(count_with_article("Hasanap"))
show_sample_verses("Hasanap")

print("\n\n=== SAYYI'A (bad deed) - ROOT:swA ===")
display(discover_by_root("swA"))
print("\nTarget lemma: say~i}ap (سيئة = bad deed/evil)")
print(count_with_article("say~i}ap"))
show_sample_verses("say~i}ap")

=== HASANA (good deed) - ROOT:Hsn ===


,LEM,POS,count,arabic
0,muHosin,N,38,مُحْسِن
1,>aHosan,N,35,أَحْسَن
2,Hasanap,N,25,حَسَنَة
3,>aHosana,V,21,أَحْسَنَ
4,Hasan,ADJ,20,حَسَن
5,Huson,N,13,حُسْن
6,<iHosa`n,N,12,إِحْسَٰن
7,HusonaY`,N,10,حُسْنَىٰ
8,HusonaY`,ADJ,7,حُسْنَىٰ
9,Hasana`t,N,3,حَسَنَٰت



Target lemma: Hasanap (حسنة = good deed)
{'total': 28, 'with_al': 11, 'without_al': 17, 'by_pos': {'N': 25, 'ADJ': 3}, 'by_number': {}}


  (  2:201:8)      form=HasanapF [حَسَنَةً] POS=N
  (  2:201:11)      form=HasanapF [حَسَنَةً] POS=N
  (  3:120:3)      form=HasanapN [حَسَنَةٌ] POS=N
  (  4: 40:9)      form=HasanapF [حَسَنَةً] POS=N
  (  4: 78:12)      form=HasanapN [حَسَنَةٌ] POS=N


=== SAYYI'A (bad deed) - ROOT:swA ===


,LEM,POS,count,arabic
0,suw^',N,50,سُو^ء
1,say~i_#aAt,N,36,سَيِّـ#َات
2,saA^'a,V,30,سَا^ءَ
3,say~i}ap,N,21,سَيِّئَة
4,sawo',N,9,سَوْء
5,>asaA^'a,V,5,أَسَا^ءَ
6,sawo'a`t,N,5,سَوْءَٰت
7,say~i},ADJ,3,سَيِّئ
8,>asowa>,N,2,أَسْوَأ
9,sawo'ap,N,2,سَوْءَة



Target lemma: say~i}ap (سيئة = bad deed/evil)
{'total': 22, 'with_al': 10, 'without_al': 12, 'by_pos': {'N': 21, 'ADJ': 1}, 'by_number': {}}
  (  2: 81:4)      form=say~i}apF [سَيِّئَةً] POS=N
  (  3:120:7)      form=say~i}apN [سَيِّئَةٌ] POS=N
  (  4: 78:20)      form=say~i}apN [سَيِّئَةٌ] POS=N
  (  4: 79:10)      form=say~i}apK [سَيِّئَةٍ] POS=N
  (  4: 85:12)      form=say~i}apF [سَيِّئَةً] POS=ADJ


In [18]:
# --- 3.14 Proper nouns: Adam, Isa/Jesus, Insan (human), Iblis ---
print("=== ADAM - proper noun, no ROOT ===")
display(discover_no_root("A\\^dam"))
print(count_with_article("A^dam"))
show_sample_verses("A^dam")

print("\n\n=== ISA/JESUS - proper noun, no ROOT ===")
display(discover_no_root("EiysaY"))
print(count_with_article("EiysaY"))
show_sample_verses("EiysaY")

print("\n\n=== INSAN (human) - ROOT:Ans ===")
display(discover_by_root("Ans"))
print("\nTarget lemma: <insa`n (إنسان = human)")
print(count_with_article("<insa`n"))
show_sample_verses("<insa`n")

print("\n\n=== IBLIS - proper noun, no ROOT ===")
display(discover_no_root("<iboliys"))
print(count_with_article("<iboliys"))
show_sample_verses("<iboliys")

=== ADAM - proper noun, no ROOT ===


,LEM,POS,count,arabic
0,A^dam,PN,25,ا^دَم


{'total': 25, 'with_al': 0, 'without_al': 25, 'by_pos': {'PN': 25}, 'by_number': {}}
  (  2: 31:2)      form='aAdama [ءَادَمَ] POS=PN
  (  2: 33:2)      form=_#aAdamu [ـ#َادَمُ] POS=PN
  (  2: 34:5)      form='aAdama [ءَادَمَ] POS=PN
  (  2: 35:2)      form=_#aAdamu [ـ#َادَمُ] POS=PN
  (  2: 37:2)      form='aAdamu [ءَادَمُ] POS=PN


=== ISA/JESUS - proper noun, no ROOT ===


,LEM,POS,count,arabic
0,EiysaY,PN,25,عِيسَى


{'total': 25, 'with_al': 0, 'without_al': 25, 'by_pos': {'PN': 25}, 'by_number': {}}
  (  2: 87:10)      form=EiysaY [عِيسَى] POS=PN
  (  2:136:18)      form=EiysaY` [عِيسَىٰ] POS=PN
  (  2:253:15)      form=EiysaY [عِيسَى] POS=PN
  (  3: 45:12)      form=EiysaY [عِيسَى] POS=PN
  (  3: 52:3)      form=EiysaY` [عِيسَىٰ] POS=PN


=== INSAN (human) - ROOT:Ans ===


...


Target lemma: <insa`n (إنسان = human)


{'total': 71, 'with_al': 64, 'without_al': 7, 'by_pos': {'N': 71}, 'by_number': {'P': 6}}
  (  2: 60:17)      form=>unaAsK [أُنَاسٍ] POS=N
  (  4: 28:7) AL+ form=<insa`nu [إِنسَٰنُ] POS=N
  (  7: 82:12)      form=>unaAsN [أُنَاسٌ] POS=N
  (  7:160:24)      form=>unaAsK [أُنَاسٍ] POS=N
  ( 10: 12:3) AL+ form=<insa`na [إِنسَٰنَ] POS=N


=== IBLIS - proper noun, no ROOT ===


,LEM,POS,count,arabic
0,<iboliys,PN,11,إِبْلِيس


{'total': 11, 'with_al': 0, 'without_al': 11, 'by_pos': {'PN': 11}, 'by_number': {}}


  (  2: 34:8)      form=<iboliysa [إِبْلِيسَ] POS=PN
  (  7: 11:12)      form=<iboliysa [إِبْلِيسَ] POS=PN
  ( 15: 31:2)      form=<iboliysa [إِبْلِيسَ] POS=PN
  ( 15: 32:2)      form=<iboliysu [إِبْلِيسُ] POS=PN
  ( 17: 61:8)      form=<iboliysa [إِبْلِيسَ] POS=PN


In [19]:
# --- 3.15 Iman (belief) & Kufr (disbelief) ---
print("=== IMAN (belief) - ROOT:Amn ===")
display(discover_by_root("Amn"))
print("\nTarget lemma: <iyma`n (إيمان = belief/faith)")
print(count_with_article("<iyma`n"))
show_sample_verses("<iyma`n")

print("\n\n=== KUFR (disbelief) - ROOT:kfr ===")
display(discover_by_root("kfr"))
print("\nTarget lemma: kufor (كفر = disbelief)")
print(count_with_article("kufor"))
show_sample_verses("kufor")

=== IMAN (belief) - ROOT:Amn ===


,LEM,POS,count,arabic
0,'aAmana,V,537,ءَامَنَ
1,mu&omin,N,195,مُؤْمِن
2,<iyma`n,N,45,إِيمَٰن
3,>amina,V,20,أَمِنَ
4,m~u&omina`t,N,18,مُّؤْمِنَٰت
5,>amiyn,ADJ,14,أَمِين
6,'aAminiyn,N,10,ءَامِنِين
7,mu&omin,ADJ,7,مُؤْمِن
8,>amon,N,5,أَمْن
9,>ama`na`t,N,4,أَمَٰنَٰت



Target lemma: <iyma`n (إيمان = belief/faith)


{'total': 45, 'with_al': 17, 'without_al': 28, 'by_pos': {'N': 45}, 'by_number': {}}
  (  2: 93:24)      form=<iyma`nu [إِيمَٰنُ] POS=N
  (  2:108:14) AL+ form=<iyma`ni [إِيمَٰنِ] POS=N
  (  2:109:10)      form=<iyma`ni [إِيمَٰنِ] POS=N
  (  2:143:40)      form=<iyma`na [إِيمَٰنَ] POS=N
  (  3: 86:7)      form=<iyma`ni [إِيمَٰنِ] POS=N


=== KUFR (disbelief) - ROOT:kfr ===


,LEM,POS,count,arabic
0,kafara,V,289,كَفَرَ
1,ka`firuwn,N,119,كَٰفِرُون
2,kufor,N,37,كُفْر
3,kaAfir,N,27,كَافِر
4,kaf~ara,V,14,كَفَّرَ
5,ka`firuwn,ADJ,10,كَٰفِرُون
6,kafuwr,N,6,كَفُور
7,kafuwr,ADJ,6,كَفُور
8,kaf~a`rap,N,4,كَفَّٰرَة
9,kaf~aAr,ADJ,4,كَفَّار



Target lemma: kufor (كفر = disbelief)
{'total': 37, 'with_al': 16, 'without_al': 21, 'by_pos': {'N': 37}, 'by_number': {}}


  (  2: 88:7)      form=kufori [كُفْرِ] POS=N
  (  2: 93:19)      form=kufori [كُفْرِ] POS=N
  (  2:108:13) AL+ form=kufora [كُفْرَ] POS=N
  (  2:217:15)      form=kuforN[ [كُفْرٌ[] POS=N
  (  3: 52:5) AL+ form=kufora [كُفْرَ] POS=N


In [20]:
# --- 3.16 Embryology: Turab, Nutfa, Alaqa, Mudgha, Izam, Lahm ---
print("=== TURAB (dust) - ROOT:trb ===")
display(discover_by_root("trb"))
print("\nTarget lemma: turaAb (تراب = dust)")
print(count_with_article("turaAb"))
show_sample_verses("turaAb")

print("\n\n=== NUTFA (sperm drop) - ROOT:nTf ===")
display(discover_by_root("nTf"))
print("\nTarget lemma: n~uTofap (نطفة = sperm drop)")
print(count_with_article("n~uTofap"))
show_sample_verses("n~uTofap")

print("\n\n=== ALAQA (clot) - ROOT:Elq ===")
display(discover_by_root("Elq"))
print("\nTarget lemma: Ealaqap (علقة = clinging clot)")
print(count_with_article("Ealaqap"))
show_sample_verses("Ealaqap")
print("\nAlso: Ealaq (علق = clot, generic form)")
print(f"  count: {len(df[df['LEM'] == 'Ealaq'])}")

print("\n\n=== MUDGHA (lump) - ROOT:mDg ===")
display(discover_by_root("mDg"))
print("\nTarget lemma: muDogap (مضغة = lump of flesh)")
print(count_with_article("muDogap"))
show_sample_verses("muDogap")

print("\n\n=== IZAM (bones) - ROOT:EZm ===")
display(discover_by_root("EZm"))
print("\nTarget lemma: EaZom (عظم = bone, singular)")
print(count_with_article("EaZom"))
show_sample_verses("EaZom")
# AUDIT CORRECTION: the corpus tags the plural عظام (bones) under LEM:EaZiym (the 'great'
# lemma) as POS:N + MP entries — e.g. 23:14 EiZa`mFA twice. Counting LEM:EaZom alone gives 2
# and misses all 13 plural occurrences. Cross-checked against corpus.quran.com (root EZm).
bones_pl = df[(df["LEM"] == "EaZiym") & (df["POS"] == "N") & (df["NUMBER"] == "P")]
print(f"\nPlural عظام tagged under LEM:EaZiym (POS:N, MP): {len(bones_pl)}")
for _, row in bones_pl.iterrows():
    print(f"  ({row['chapter']:>3}:{row['verse']:>3}:{row['word']}) form={row['form']} [{bw_to_arabic(row['form'])}]")
print(f"Bones total = {len(df[df['LEM'] == 'EaZom'])} singular + {len(bones_pl)} plural = "
      f"{len(df[df['LEM'] == 'EaZom']) + len(bones_pl)}")
print("\nEaZiym (عظيم = great/mighty) itself, excluding the bones entries:")
print(f"  count: {len(df[df['LEM'] == 'EaZiym']) - len(bones_pl)}")

print("\n\n=== LAHM (flesh) - ROOT:lHm ===")
display(discover_by_root("lHm"))
print("\nTarget lemma: laHom (لحم = flesh/meat)")
print(count_with_article("laHom"))
show_sample_verses("laHom")

=== TURAB (dust) - ROOT:trb ===


,LEM,POS,count,arabic
0,turaAb,N,17,تُرَاب
1,>atoraAb,N,2,أَتْرَاب
2,>atoraAb,ADJ,1,أَتْرَاب
3,matorabap,N,1,مَتْرَبَة
4,t~araA^}ib,N,1,تَّرَا^ئِب



Target lemma: turaAb (تراب = dust)


{'total': 17, 'with_al': 1, 'without_al': 16, 'by_pos': {'N': 17}, 'by_number': {}}


  (  2:264:23)      form=turaAbN [تُرَابٌ] POS=N
  (  3: 59:10)      form=turaAbK [تُرَابٍ] POS=N
  ( 13:  5:7)      form=tura`bFA [تُرَٰبًا] POS=N
  ( 16: 59:15) AL+ form=t~uraAbi [تُّرَابِ] POS=N
  ( 18: 37:10)      form=turaAbK [تُرَابٍ] POS=N


=== NUTFA (sperm drop) - ROOT:nTf ===


...

  count: 107


=== LAHM (flesh) - ROOT:lHm ===


,LEM,POS,count,arabic
0,laHom,N,12,لَحْم



Target lemma: laHom (لحم = flesh/meat)


{'total': 12, 'with_al': 0, 'without_al': 12, 'by_pos': {'N': 12}, 'by_number': {'P': 1}}


  (  2:173:6)      form=laHoma [لَحْمَ] POS=N
  (  2:259:56)      form=laHomFA [لَحْمًا] POS=N
  (  5:  3:5)      form=laHomu [لَحْمُ] POS=N
  (  6:145:20)      form=laHoma [لَحْمَ] POS=N
  ( 16: 14:7)      form=laHomFA [لَحْمًا] POS=N


## 4. Final Counts

**Counting methods (applied uniformly to EVERY word):**
1. **Lemma** — exact lemma match, all grammatical forms (sg/dual/pl, all cases)
2. **Lemma + variants** — same as above plus irregular-plural / variant-noun selectors that the
   corpus stores under a different lemma. A selector is either a plain lemma string or a
   `(lemma, POS, NUMBER)` tuple — the tuple form is needed for bones, whose plural عظام is
   tagged `LEM:EaZiym, POS:N, MP` (see audit in notebook 03)
3. **Singular only** — lemma match, excluding dual and plural forms
4. **Root (nominal)** — all nouns + adjectives + proper nouns + time adverbs sharing the trilateral root

All four methods are reported for every word. No cherry-picking. The complete grid
(per-POS, per-number, root-by-POS including verbs, occurrence-level index) lives in
`03_audit_and_full_recount.ipynb` and `output/full_counts.csv` / `output/occurrences.csv`.

In [21]:
# For each word: define primary lemma(s), variant selectors, root, and optional gender filter.
# This is the ONLY place counting logic is defined — applied uniformly below.
#
# variants: irregular plurals / variant nouns stored under a different lemma. Either a plain
#   lemma string, or a (lemma, POS, NUMBER) tuple when only a slice of another lemma belongs
#   to this word — needed for Izam, whose plural عظام is tagged LEM:EaZiym + POS:N + MP
#   (audit evidence in notebook 03, cross-validated against corpus.quran.com).
#
# gender_filter: if set (e.g. "F"), only count entries whose PGN contains that gender.
#   Used ONLY when a single lemma conflates genuinely different words distinguishable by gender.
#   Akhira is the only such case: masc = "last/latter", fem = "the hereafter" (different words).
#   Verified: no other lemma in our list has this issue.

WORDS = [
    # (name, arabic, primary_lemmas, variants, root, gender_filter, flags)
    ("Dunya", "دنيا", ["d~unoyaA"], [], "dnw", None, ""),
    ("Akhira", "آخرة", ["A^xir"], [], "Axr", "F",
     "gender_filter=F: masc A^xir='last/latter'(40x) and fem A^xirap='the hereafter'(115x) "
     "are different words sharing a lemma (155 total). Only feminine counted."),
    ("Malak (angel)", "ملك", ["malak"], [], "mlk", None,
     "Includes ملائكة (73 pl + 2 dual). Root mlk also has malik(king), mulk(dominion) — different lemmas, not included."),
    ("Shaytan", "شيطان", ["$ayoTa`n"], [], "$Tn", None, "Includes شياطين (18 pl)."),
    ("Hayat", "حياة", ["Hayaw`p"], ["m~aHoyaA", "HayawaAn"], "Hyy", None,
     "maHyaA(2x) and HayawaAn(1x, 29:64 'the true life') are variant nouns of 'life' — "
     "symmetric with Mawt's variants (6:162 pairs maHyaA with mamaAt)."),
    ("Mawt", "موت", ["mawot"], ["mawotat", "mamaAt"], "mwt", None,
     "mawotat(3x) and mamaAt(3x) are variant noun forms of 'death'."),
    ("Rajul", "رجل", ["rajul"], ["rijaAl"], "rjl", None,
     "rijaAl(28x) = broken plural (men). rijol(15x) = feet, different word."),
    ("Imra'a", "امرأة", ["{mora>at"], ["nisaA^'"], "mrA", None,
     "nisaa'(59x, incl. niswa 12:30) is the suppletive plural (women). Different root (nsw) — normal in Arabic."),
    ("Shahr", "شهر", ["$ahor"], [], "$hr", None, "Includes أشهر/شهور (7 pl + 2 dual)."),
    ("Yawm", "يوم", ["yawom"], [], "ywm", None,
     "Includes أيام (27 pl) + يومين (3 dual). yawma'idhin (68x) is a separate lemma, not counted."),
    ("Bahr", "بحر", ["baHor"], [], "bHr", None, ""),
    ("Barr", "بر", ["bar~"], [], "brr", None,
     "Lemma bar~ = land(12x, all 'in the land') + dutiful(2x: 19:14, 19:32) + divine name "
     "al-Barr(1x: 52:28) + righteous pl. أبرار/بررة(7x). Form-level split in notebook 03."),
    ("Jannah", "جنة", ["jan~ap"], [], "jnn", None,
     "Includes جنات (71 pl + 8 dual). Root jnn also has jinn, majnuun — different lemmas, not included."),
    ("Jahannam", "جهنم", ["jahan~am"], [], None, None, "Proper noun, no root in corpus."),
    ("Harr", "حر", ["Har~"], ["Haruwr"], "Hrr", None,
     "Haruwr(1x) = scorching heat. Hur~(2x) = free man, different word, excluded."),
    ("Bard", "برد", ["barod"], ["baArid"], "brd", None,
     "baArid(2x) = cold (adj). barad(1x) = hail, excluded."),
    ("Zakat", "زكاة", ["zakaw`p"], [], "zkw", None, ""),
    ("Baraka", "بركة", ["baraka`t"], ["mubaArak", "m~uba`rakap"], "brk", None,
     "Singular بركة does not occur; barakat(3x) is the plural. mubaArak/muba`rakap = blessed "
     "(adj/noun). Verbs ba`raka/tabaAraka excluded."),
    ("Qaala", "قال", ["qaAla"], [], "qwl", None, "Verb (1618 total)."),
    ("Adhab", "عذاب", ["Ea*aAb"], [], "E*b", None, ""),
    ("Rahma", "رحمة", ["raHomap"], [], "rHm", None, ""),
    ("Maghfira", "مغفرة", ["m~agofirap"], ["guforaAn"], "gfr", None,
     "guforaAn(1x) = forgiveness (synonym)."),
    ("Ghani", "غني", ["ganiY~"], [], "gny", None, ""),
    ("Faqir", "فقير", ["faqiyr"], [], "fqr", None, ""),
    ("Hasana", "حسنة", ["Hasanap"], ["Hasana`t"], "Hsn", None, ""),
    ("Sayyi'a", "سيئة", ["say~i}ap"], ["say~i_#aAt"], "swA", None, ""),
    ("Adam", "آدم", ["A^dam"], [], None, None, "Proper noun, no root."),
    ("Isa", "عيسى", ["EiysaY"], [], None, None, "Proper noun, no root."),
    ("Insan", "إنسان", ["<insa`n"], ["<ins"], "Ans", None,
     "Lemma includes أناس/أناسي (6 pl). <ins(18x) = mankind (collective noun)."),
    ("Iblis", "إبليس", ["<iboliys"], [], None, None, "Proper noun, no root."),
    ("Iman", "إيمان", ["<iyma`n"], [], "Amn", None, ""),
    ("Kufr", "كفر", ["kufor"], ["kuforaAn", "kufuwr"], "kfr", None,
     "kuforaAn(1x) and kufuwr(3x) are variant noun forms."),
    ("Turab", "تراب", ["turaAb"], [], "trb", None, ""),
    ("Nutfa", "نطفة", ["n~uTofap"], [], "nTf", None, ""),
    ("Alaqa", "علقة", ["Ealaqap"], ["Ealaq"], "Elq", None, ""),
    ("Mudgha", "مضغة", ["muDogap"], [], "mDg", None, ""),
    ("Izam", "عظم", ["EaZom"], [("EaZiym", "N", "P")], "EZm", None,
     "CORRECTED: plural عظام (13x, incl. 23:14 twice) is tagged LEM:EaZiym POS:N+MP — the "
     "selector picks exactly those. EaZiym minus bones = great/mighty (107x), different word."),
    ("Lahm", "لحم", ["laHom"], [], "lHm", None, ""),
]
assert len(WORDS) == 38

def variant_mask(v):
    """A variant is a lemma string or a (lemma, POS, NUMBER) tuple; None = no constraint."""
    if isinstance(v, str):
        return df["LEM"] == v
    lem, pos, number = v
    m = df["LEM"] == lem
    if pos is not None:
        m &= df["POS"] == pos
    if number is not None:
        m &= df["NUMBER"] == number
    return m

# ---- Count everything uniformly ----
rows = []
for name, arabic, primary_lems, variants, root, gender_filter, flags in WORDS:
    lem_mask = df["LEM"].isin(primary_lems)
    if gender_filter:
        lem_mask = lem_mask & df["PGN"].str.contains(gender_filter, na=False)
    lem_count = int(lem_mask.sum())

    combined_mask = lem_mask.copy()
    for v in variants:
        combined_mask |= variant_mask(v)
    combined_count = int(combined_mask.sum())

    sg_mask = lem_mask & ~df["NUMBER"].isin(["D", "P"])
    sg_count = int(sg_mask.sum())

    if root:
        root_nom_mask = (df["ROOT"] == root) & df["POS"].isin(["N", "ADJ", "PN", "T"])
        root_nom_count = int(root_nom_mask.sum())
    else:
        root_nom_count = None

    rows.append({
        "Word": name,
        "Arabic": arabic,
        "BW Lemma": ", ".join(primary_lems),
        "Lemma": lem_count,
        "Lemma+Variants": combined_count,
        "SingularOnly": sg_count,
        "RootNominal": root_nom_count if root_nom_count is not None else "—",
        "Flags": flags if flags else "",
    })

result = pd.DataFrame(rows)
print("COUNTS (4 methods, applied uniformly to every word):\n")
display(result[["Word", "Arabic", "BW Lemma", "Lemma", "Lemma+Variants", "SingularOnly", "RootNominal"]])

flagged = result[result["Flags"] != ""]
if len(flagged):
    print("\n\nNOTES:\n")
    for _, row in flagged.iterrows():
        print(f"  {row['Word']} ({row['Arabic']}): {row['Flags']}\n")

COUNTS (4 methods, applied uniformly to every word):



,Word,Arabic,BW Lemma,Lemma,Lemma+Variants,SingularOnly,RootNominal
0,Dunya,دنيا,d~unoyaA,115,115,115,131
1,Akhira,آخرة,A^xir,115,115,115,226
2,Malak (angel),ملك,malak,88,88,13,162
3,Shaytan,شيطان,$ayoTa`n,88,88,70,88
4,Hayat,حياة,Hayaw`p,76,79,76,113
5,Mawt,موت,mawot,50,56,50,105
6,Rajul,رجل,rajul,29,57,24,73
7,Imra'a,امرأة,{mora>at,26,85,24,38
8,Shahr,شهر,$ahor,21,21,12,21
9,Yawm,يوم,yawom,405,405,375,405




NOTES:

  Akhira (آخرة): gender_filter=F: masc A^xir='last/latter'(40x) and fem A^xirap='the hereafter'(115x) are different words sharing a lemma (155 total). Only feminine counted.

  Malak (angel) (ملك): Includes ملائكة (73 pl + 2 dual). Root mlk also has malik(king), mulk(dominion) — different lemmas, not included.

  Shaytan (شيطان): Includes شياطين (18 pl).

  Hayat (حياة): maHyaA(2x) and HayawaAn(1x, 29:64 'the true life') are variant nouns of 'life' — symmetric with Mawt's variants (6:162 pairs maHyaA with mamaAt).

  Mawt (موت): mawotat(3x) and mamaAt(3x) are variant noun forms of 'death'.

  Rajul (رجل): rijaAl(28x) = broken plural (men). rijol(15x) = feet, different word.

  Imra'a (امرأة): nisaa'(59x, incl. niswa 12:30) is the suppletive plural (women). Different root (nsw) — normal in Arabic.

  Shahr (شهر): Includes أشهر/شهور (7 pl + 2 dual).

  Yawm (يوم): Includes أيام (27 pl) + يومين (3 dual). yawma'idhin (68x) is a separate lemma, not counted.

  Barr (بر): Lemma bar